## Setup

### Load Modules

In [ ]:
%load_ext autoreload
%autoreload 2

#General Import
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle
from os.path import join
from sklearn.model_selection import KFold
from matplotlib import colormaps as cmaps
from mne.filter import filter_data, resample
import scipy.stats as stats
import pandas as pd
from itertools import product
import xarray as xr
from scipy.signal import coherence, welch
from matplotlib import cm
from matplotlib.ticker import LinearLocator
import re
import colorcet as cc
from wordfreq import word_frequency

#ML Import
from sklearn.decomposition import PCA, FastICA, SparsePCA, FactorAnalysis, NMF
from sklearn.preprocessing import StandardScaler, scale
from sklearn.metrics import silhouette_score
from dtaidistance.preprocessing import differencing
import dtaidistance.clustering.kmeans as dtwkmeans
import dtaidistance as dta
import jPCA
import scipy.signal as signal
from statsmodels.tsa.stattools import grangercausalitytests
import tslearn

#Electrophysiology Import
from spyeeg.models.TRF import TRFEstimator
from spyeeg.models.ERP import ERP_class
from spyeeg.utils import lag_matrix
import mne
import frites
from frites.simulations import sim_multi_suj_ephy
from frites.dataset import DatasetEphy
from frites.workflow import WfConnComod
from frites import set_mpl_style
import frites.conn as conn
from scipy.signal import welch
import spectral_connectivity 

#Graph Import
import networkx as nx
from matplotlib_venn import venn3, venn3_circles
import matplotlib.gridspec as gridspec

#Performance Import
import time
import psutil

#Local Import
from stats_utils import cliffs_delta, cohen_d
from nice_utils import estimate_loop_time, decorator_loop
from preprocessing_utils import mono_to_bipolar, select_channels, available_regions, delete_channels, adj_scale
from signal_utils import sparse_resample, lag_finder
from viz_utils import create_matshow_gif, create_collection_gif, _arrow3D


### Load Features

In [ ]:
# Acoustic Regressors
fs = 100
new_path = 'C:/Users/D-CAP/Documents/GitHub/witching-star/regressors/selected_regs.pkl'
new_data = pickle.load(open(new_path, 'rb'))
data_fs = new_data['fs']
new_regressors = new_data['regs']
new_names = new_data['regs_name']
ratio = fs/data_fs
new_duration = int(new_regressors.shape[0] * ratio) + 1

new_resamp = []
for i in range(new_regressors.shape[1]):
    name = new_names[i]
    if name in ['Intensity', 'Envelope Oganian', 'Envelope Derivative TF', 'F0 Loudness', 'SpectralFlux Filtered', 'SpectralFlux not_filtered']:
        new_reg = mne.filter.resample(new_regressors[:,i], up=100, down=data_fs)[:new_duration]
    elif name in ['peakEnv_tf', 'Syllabe Onset', 'p-syl', 'Phono']:
        new_reg = new_regressors[:,i] - np.min(new_regressors[:,i])
        new_reg = sparse_resample(new_reg, new_fs = fs, current_fs = data_fs)[:new_duration]
    else:
        print('wut')
    new_resamp.append(new_reg)
new_resamp = np.asarray(new_resamp).T
regressors = new_resamp
regressors_name = new_data['regs_name']

In [ ]:
# Renyi2 Array

path_renyi = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/renyi_array2.pickle"
renyi_data = pickle.load(open(path_renyi, 'rb'))
data_fs = renyi_data['fs']
X = np.roll(renyi_data['X'],4, axis=0)
onsets = np.where(X[:,5] >0)[0]
reg_renyi = np.zeros([regressors.shape[0], X.shape[1]])

for onset in onsets:
    new_onset = int(onset/data_fs*fs)
    for renyi_index in range(X.shape[1]):
        reg_renyi[new_onset, renyi_index] = X[onset,renyi_index]

regressors = np.hstack([regressors, reg_renyi])
regressors_name = regressors_name + renyi_data['names']
renyi_values_wrd = [float(renyi_name.split('renyi ')[1]) for renyi_name in renyi_data['names'][:20]]

### Load Neural Data

In [ ]:
# Load Broadband

data_subject = dict()
channels_subject = dict()
locations_subject = dict()
path_data = "D:/DataSEEG_Sorciere/BIDS/data_mne_fif"
for index_subject in range(1,40):
    try:
        name_subject = 'sub-{:03}'.format(index_subject)
        name_file = name_subject + '_task-iSpeech_speech-epo.fif'
        path_file = os.path.join(path_data, name_subject,'preprocessed','epochs',name_file)
        if os.path.isfile(path_file):
            mne_data = mne.read_epochs(path_file, verbose = False)
        elif os.path.isfile(os.path.join(path_data, name_subject,'preprocessed','monopolar',name_file)):
            path_file = os.path.join(path_data, name_subject,'preprocessed','monopolar',name_file)
            mne_data = mne.read_epochs(path_file, verbose = False)
        else:
            path_file = os.path.join(path_data, name_subject,'preprocessed','epochs','monopolar',name_file)
            mne_data = mne.read_epochs(path_file, verbose = False)
        mne_data_resample = mne_data.resample(fs, npad = 'auto', verbose = False)
        mne_data_resample.filter(0.3, 49, verbose = False)
        channels = mne_data_resample.ch_names
        montage = mne_data_resample.get_montage()
        ch_names = [n for n,_ in montage.get_positions()['ch_pos'].items()] 
        loc = (1e3 * np.stack([coord for _,coord in montage.get_positions()['ch_pos'].items()])).T  # store locations
        
        data_subject[index_subject] = mne_data_resample.get_data(copy = False)[0].T[:regressors.shape[0],:]
        channels_subject[index_subject] = channels
        locations_subject[index_subject] = loc
        print('Subject', index_subject, 'loaded')
    except:
        print('Error in subject', index_subject)

data_bipolar, channels_bipolar, locations_bipolar = mono_to_bipolar(data_subject, channels_subject, locations_subject)

Subject 1 loaded
Subject 2 loaded
Subject 3 loaded
Subject 4 loaded
Subject 5 loaded
Subject 6 loaded
Subject 7 loaded
Subject 8 loaded
Subject 9 loaded
Subject 10 loaded
Subject 11 loaded
Subject 12 loaded
Subject 13 loaded
Subject 14 loaded
Subject 15 loaded
Subject 16 loaded
Subject 17 loaded
Subject 18 loaded
Subject 19 loaded
Subject 20 loaded
Subject 21 loaded
Error in subject 22
Subject 23 loaded
Subject 24 loaded
Error in subject 25
Subject 26 loaded
Subject 27 loaded
Subject 28 loaded
Subject 29 loaded
Subject 30 loaded
Subject 31 loaded
Subject 32 loaded
Subject 33 loaded
Subject 34 loaded
Subject 35 loaded
Subject 36 loaded
Error in subject 37
Error in subject 38
Error in subject 39


In [ ]:
atlas_subject = dict()
path_data = "D:/DataSEEG_Sorciere/BIDS/data_mne_fif"
path_atlas = "D:/DataSEEG_Sorciere/BIDS/freesurfer"
for index_subject in range(1,40):
    try:
        name_subject = 'sub-{:03}'.format(index_subject)
        name_bipolar_atlas = 'elecbipolar2atlas.mat'
        name_monopolar_atlas = 'elec2atlas.mat'
        path_bipolar_atlas = os.path.join(path_atlas, name_subject,name_bipolar_atlas)
        path_monopolar_atlas = os.path.join(path_atlas, name_subject,name_monopolar_atlas)
        bipolar_atlas = get_bipolar_atlas(path_bipolar_atlas, atlas = 'Desikan_Killiany') #Destrieux
        atlas_subject[index_subject] = bipolar_atlas
    except:
        print('Atlas error in subject', index_subject)

subjects =  list(atlas_subject.keys())
new_atlas = dict()
for subject_index in atlas_subject:
    new_atlas[subject_index] = dict()
    for channels_bipolar in atlas_subject[subject_index]:
        new_name = channels_bipolar.split('-')[0] + '||' + channels_bipolar.split('-')[1]
        new_atlas[subject_index][new_name] = atlas_subject[subject_index][channels_bipolar][0]

## PID Extraction

### Load Files

In [ ]:
channel_selection = ['H','T']
#channel_selection = ['H','T','OP', 'OT', 'OR', 'OC', 'TP', 'TB', 'GPH']
channel_selection = []
exclude = False

syn_group = dict()
red_group = dict()
uni_group = dict()
tot_group = dict()
pos_group = dict()


reg_color = []
reg_color_dict = dict()
reg_label = []
reg_label_dict = dict()
montage_choice = 'bipo'
skip_symmetry = False
subjects_indices = np.arange(len(data_bipolar))

reg_pairs = ['233-253', '233-258' ,'253-258']


apply_baseline = True
baseline_str = (not apply_baseline) * 'no_baseline'
reg_pair0 = reg_pairs[0]
cat = '' #'cat_' or ''
channel_selection_join = ''.join(channel_selection)
colorm = cm.brg


syn_group, red_group, uni1_group, uni2_group, ii_group = dict(), dict(), dict(), dict(), dict()
for reg_index, regpair in enumerate(reg_pairs):
    reg1, reg2 = int(regpair.split('-')[0]), int(regpair.split('-')[1])
    filename = 'PID_reg/PID6_' + baseline_str + montage_choice + '_' + channel_selection_join + '_regpair_' + str(reg1) + '-' + str(reg2) + '.pickle'
    with open(filename, 'rb') as file:
        pid = pickle.load(file)

    regname1, regname2 = pid['regressor 1'][:9], pid['regressor 2'][:9]
    info_label = ['Unique 1', 'Unique 2','Redundancy', 'Synergy', 'Interaction']
    timecourse_array = pid['timearray']
    channels_group = pid['channels']
    color_value = int(reg_index/len(reg_pairs)*240)
    reg_color.append(colorm(color_value))
    reg_color_dict[regpair] = reg_color[reg_index]
    reg_label.append(regname1 + '-' + regname2)
    reg_label_dict[reg_index] = regname1 + '-' + regname2

    
    for subject_id in pid['data']:
        if not subject_id in syn_group:
            syn_group[subject_id],red_group[subject_id],uni1_group[subject_id],uni2_group[subject_id],ii_group[subject_id] = dict(), dict(), dict(), dict(), dict()
            
        dataset_syn = xr.Dataset({'Synergy': (['roi', 'times'], pid['data'][subject_id]['synergy'].T)},
                                coords={'roi': channels_group[subject_id],'times': timecourse_array})['Synergy']
        dataset_red = xr.Dataset({'Redundancy': (['roi', 'times'], pid['data'][subject_id]['redundancy'].T)},
                                coords={'roi': channels_group[subject_id],'times': timecourse_array})['Redundancy']
        dataset_uni1 = xr.Dataset({'Unique1': (['roi', 'times'], pid['data'][subject_id]['unique'][:,:,0].T)},
                                coords={'roi': channels_group[subject_id],'times': timecourse_array})['Unique1']
        dataset_uni2 = xr.Dataset({'Unique2': (['roi', 'times'], pid['data'][subject_id]['unique'][:,:,1].T)},
                                coords={'roi': channels_group[subject_id],'times': timecourse_array})['Unique2']
        syn_group[subject_id][regpair] = {'timearray': dataset_syn}
        red_group[subject_id][regpair] = {'timearray': dataset_red}
        uni1_group[subject_id][regpair] = {'timearray': dataset_uni1}
        uni2_group[subject_id][regpair] = {'timearray': dataset_uni2}
        ii_group[subject_id][regpair] = {'timearray': dataset_syn - dataset_red}


### Cluster Information

#### Choose Cluster

In [ ]:
cluster_none = False
regressors_clusters = [233,258]
regressors_clusters = [258]
regressors_clusters = list(np.arange(233,253)) + [258]

clustering = 'NMF6' #'NMF'


cluster_group = dict()
for subject_index, subject_id in enumerate(uni1_group):
    regressors_str = '_'.join(np.asarray(regressors_clusters).astype('str'))
    filename = 'MI_cluster/' + clustering + '_' + montage_choice + '_' +  str(subject_id) + '_' + regressors_str + '.pickle'
    with open(filename, 'rb') as file:
        cluster_group[subject_id] = pickle.load(file)

n_clusters = len(cluster_group[subject_id])
#n_clusters = 2
n_interactions = int(comb(n_clusters,2))
edges_list = []
for cluster_1 in range(n_clusters):
    for cluster_2 in range(cluster_1+1, n_clusters):
        edges_list.append(str(cluster_1) + '-' + str(cluster_2))    

#### Cluster Data

In [ ]:
uni1_cluster, uni2_cluster, syn_cluster, red_cluster, ii_cluster = dict(),dict(),dict(),dict(),dict()

for subject_index, subject_id in enumerate(uni1_group):
    uni1_cluster[subject_id], uni2_cluster[subject_id], syn_cluster[subject_id], red_cluster[subject_id],ii_cluster[subject_id] = dict(),dict(),dict(),dict(), dict()
    for regressor_index, regressor_id in enumerate(uni1_group[subject_id]):
        uni1_cluster[subject_id][regressor_id], uni2_cluster[subject_id][regressor_id], syn_cluster[subject_id][regressor_id], red_cluster[subject_id][regressor_id],ii_cluster[subject_id][regressor_id] = dict(),dict(),dict(),dict(), dict()
        for cluster in range(n_clusters):
            roi_find = cluster_group[subject_id][cluster]
            if cluster_none:
                uni1_cluster[subject_id][regressor_id][0] = {'timearray':uni1_group[subject_id][regressor_id]['timearray']}
                uni2_cluster[subject_id][regressor_id][0] = {'timearray':uni2_group[subject_id][regressor_id]['timearray']}
                red_cluster[subject_id][regressor_id][0] = {'timearray':red_group[subject_id][regressor_id]['timearray']}
                syn_cluster[subject_id][regressor_id][0] = {'timearray':syn_group[subject_id][regressor_id]['timearray']}
                ii_cluster[subject_id][regressor_id][0] = {'timearray': syn_cluster[subject_id][regressor_id][0]['timearray'] - red_cluster[subject_id][regressor_id][0]['timearray']}
            elif len(roi_find) > 0:
                uni1_cluster[subject_id][regressor_id][cluster] = {'timearray':uni1_group[subject_id][regressor_id]['timearray'].sel(roi = roi_find)}
                uni2_cluster[subject_id][regressor_id][cluster] = {'timearray':uni2_group[subject_id][regressor_id]['timearray'].sel(roi = roi_find)}
                red_cluster[subject_id][regressor_id][cluster] = {'timearray':red_group[subject_id][regressor_id]['timearray'].sel(roi = roi_find)}
                syn_cluster[subject_id][regressor_id][cluster] = {'timearray':syn_group[subject_id][regressor_id]['timearray'].sel(roi = roi_find)}
                ii_cluster[subject_id][regressor_id][cluster] = {'timearray': syn_cluster[subject_id][regressor_id][cluster]['timearray'] - red_cluster[subject_id][regressor_id][cluster]['timearray']}

## Timecourse Analysis

In [ ]:
meta_pid = dict()
cluster_choices = [0,1,2] #[0,1]

for subject_index, subject_id in enumerate(uni1_group.keys()):
    for regpair in reg_pairs:
        if not regpair in meta_pid:
            print(regpair)
            meta_pid[regpair] = {'uni1' : dict(), 'uni2' : dict(), 'red' : dict(), 'syn' : dict(), 'roi' : dict()}
        for cluster in ii_cluster[subject_id][regpair]:
            if not cluster in meta_pid[regpair]['uni1']:
                meta_pid[regpair]['uni1'][cluster], meta_pid[regpair]['uni2'][cluster], meta_pid[regpair]['red'][cluster], meta_pid[regpair]['syn'][cluster], meta_pid[regpair]['roi'][cluster] = [],[],[],[],[]
            if cluster in cluster_choices:
                uni1_data = uni1_cluster[subject_id][regpair][cluster]['timearray'].data 
                uni2_data = uni2_cluster[subject_id][regpair][cluster]['timearray'].data 
                red_data = red_cluster[subject_id][regpair][cluster]['timearray'].data #- np.repeat(red_cluster[subject_id][regpair][cluster]['timearray'].data.min(1),red_cluster[subject_id][regpair][cluster]['timearray'].data.shape[1]).reshape(red_cluster[subject_id][regpair][cluster]['timearray'].data.shape)
                syn_data = syn_cluster[subject_id][regpair][cluster]['timearray'].data #- np.repeat(syn_cluster[subject_id][regpair][cluster]['timearray'].data.min(1),syn_cluster[subject_id][regpair][cluster]['timearray'].data.shape[1]).reshape(syn_cluster[subject_id][regpair][cluster]['timearray'].data.shape)

                meta_pid[regpair]['uni1'][cluster] += (list(uni1_data))
                meta_pid[regpair]['uni2'][cluster] += (list(uni2_data))
                meta_pid[regpair]['red'][cluster] += (list(red_data))
                meta_pid[regpair]['syn'][cluster] += (list(syn_data))
                meta_pid[regpair]['roi'][cluster] += (list(uni1_cluster[subject_id][regpair][cluster]['timearray']['roi'].data))

for regpair in meta_pid:
    for info in meta_pid[regpair]:
        for cluster in meta_pid[regpair][info]:
            meta_pid[regpair][info][cluster] = np.asarray(meta_pid[regpair][info][cluster])

        

In [ ]:
%matplotlib qt
p_thres = 0.05 #0.05
start_baseline = 50 #100
end_baseline = 75 #150
start_test = 75 #150
end_test = 220 #300
wrd_distrib = np.load('wrd_distrib.npy')
baseline_samples = list(np.arange(timecourse_array.shape[0]))[start_baseline:end_baseline]
fig, axes = plt.subplots(2,4, figsize = (12,6), sharey = True, sharex = True)

color_list =['b', 'r'] 

for regpair_index, regpair in enumerate(meta_pid):
    print(regpair)
    for info_index, info in enumerate(list(meta_pid[regpair].keys())[:-1]):
        for cluster in meta_pid[regpair][info]:
            if (regpair_index == 0) or (regpair_index - 1 == cluster):
                ax = axes[min(regpair_index,1),info_index]
                data = meta_pid[regpair][info][cluster]
                limit = np.percentile(data[:,start_baseline:end_baseline].mean(0),95)
                baseline = data[:,baseline_samples]
                baseline_avg, baseline_std = np.mean(baseline,axis = 1), np.std(baseline,axis = 1)
                baseline_std = np.clip(baseline_std,5e-5,np.inf)
                baseline_std = np.clip(baseline_std,5e-5,np.inf)

                normalized_data = data - data[:,50:].mean(0).min()

                statistics1 = []
                for i in range(len(timecourse_array)):

                    pval = stats.ttest_rel(data[:,i], baseline.mean(1), alternative = 'greater')[1]
                    statistics1.append(pval)
                statistics1 = list(np.ones(start_test)) + list(multipletests(statistics1[start_test:end_test], method = 'fdr_tsbh')[1]) + list(np.ones(len(statistics1) - end_test))
                statistics = np.asarray(statistics1) < p_thres
                statistics = filter_binary(statistics, min_cluster_size= 15)
                non_statistics = np.asarray(statistics1) > p_thres
                starts = np.where(np.diff(np.concatenate(([0], statistics))) == 1)[0]
                ends   = np.where(np.diff(np.concatenate((statistics, [0]))) == -1)[0]

                ax.plot(timecourse_array,normalized_data.mean(0), color = 'grey', alpha = 0.3,linewidth = 2)
                ax.fill_between(timecourse_array,
                                (normalized_data.mean(0) - normalized_data.std(0) / np.sqrt(normalized_data.shape[0])),
                                (normalized_data.mean(0) + normalized_data.std(0) / np.sqrt(normalized_data.shape[0])),
                                alpha = 0.3, color = 'grey')
                for s, e in zip(starts, ends):
                    ax.plot(timecourse_array[s:e],normalized_data.mean(0)[s:e], color = color_list[cluster], linewidth = 2) 
                    ax.fill_between(timecourse_array[s:e],
                                    (normalized_data.mean(0) - normalized_data.std(0) / np.sqrt(normalized_data.shape[0]))[s:e],
                                    (normalized_data.mean(0) + normalized_data.std(0) / np.sqrt(normalized_data.shape[0]))[s:e],
                                    alpha = 0.3, color = color_list[cluster])
                

                ax.axvline(0,color = 'k', lw = 2, ls = '--')
                ax.axvline(np.percentile(wrd_distrib,95),color = 'k', lw = 2, ls = '--')


                ax.spines[['right', 'top']].set_visible(0)
                ax.spines[['bottom', 'left']].set_linewidth(2)
                ax.tick_params(width=3, labelsize = 16)
                ax.set_xticks([-1,-0.5,0,0.5,1])
                ax.set_xlim(-0.3,1.2)
                fig.patch.set_facecolor((0.941, 0.969, 1.0))
                ax.set_facecolor((0.941, 0.969, 1.0))
                #for s, e in zip(starts, ends):
                #    ax.plot(timecourse_array[[s, e]],[[-0.5,-0.5], [-1,-1]][cluster],color = ['b','r'][cluster], alpha = 1, lw = 3)




axes[0,0].set_title('Unique\nDispersion', size = 16)
axes[0,1].set_title('Unique\nStrength', size = 16)
axes[0,2].set_title('Dispersion-Strength\nRedundancy', size = 16)
axes[0,3].set_title('Dispersion-Strength\nSynergy', size = 16)

axes[1,0].set_title('\n\n\n\nUnique\nUncertainty', size = 16)
axes[1,1].set_title('\n\n\n\nUnique\nSurprisal', size = 16)
axes[1,2].set_title('\n\n\n\nUncertainty-Surprisal\nRedundancy', size = 16)
axes[1,3].set_title('\n\n\n\nUncertainty-Surprisal\nSynergy', size = 16)

axes[0,0].set_ylabel('Information\n(bits)', size = 16)
axes[1,0].set_ylabel('Information\n(bits)', size = 16)

axes[0,1].set_xlabel("                                   Time (s)", size = 16)
axes[1,1].set_xlabel("                                   Time (s)", size = 16)

ymin_uni, ymax_uni = np.max([axes[0,0].get_ylim()[0],axes[0,1].get_ylim()[0],axes[1,0].get_ylim()[0],axes[1,1].get_ylim()[0]]), np.max([axes[0,0].get_ylim()[1],axes[0,1].get_ylim()[1],axes[1,0].get_ylim()[1],axes[1,1].get_ylim()[1]])
axes[0,0].set_ylim([ymin_uni, ymax_uni])

fig.tight_layout()
